In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect('../data/bike.db')
def q(sql):
    return pd.read_sql(sql, con)

q("""
SELECT strftime('%m', 대여일자) AS 월,
       COUNT(*)            AS 일수,
       SUM(이용건수)        AS 총이용,
       ROUND(AVG(이용건수)) AS 일평균,
       ROUND(AVG(평균기온), 1) AS 평균기온
FROM daily
GROUP BY 월
ORDER BY 월
""")

,월,일수,총이용,일평균,평균기온
0,01,31,1595783,51477.0,-3.6
1,02,28,1744655,62309.0,2.7
2,03,31,2842167,91683.0,8.2
3,04,30,3824157,127472.0,15.0
4,05,31,3905041,125969.0,19.6
5,06,30,4066539,135551.0,24.0


In [2]:
q("""
SELECT 요일,
       COUNT(*) AS 일수,
       ROUND(AVG(이용건수)) AS 일평균
FROM daily
GROUP BY 요일
ORDER BY CASE 요일
    WHEN 'Monday' THEN 1 WHEN 'Tuesday' THEN 2 WHEN 'Wednesday' THEN 3
    WHEN 'Thursday' THEN 4 WHEN 'Friday' THEN 5 WHEN 'Saturday' THEN 6
    ELSE 7 END
""")

,요일,일수,일평균
0,Monday,26,98299.0
1,Tuesday,26,107330.0
2,Wednesday,25,104073.0
3,Thursday,26,100726.0
4,Friday,26,109462.0
5,Saturday,26,92340.0
6,Sunday,26,83248.0


In [3]:
q("""
WITH base AS (
    SELECT *,
           strftime('%m', 대여일자) AS 월,
           CASE WHEN 요일 IN ('Saturday','Sunday')
                  OR date(대여일자) IN ('2026-01-01','2026-02-16','2026-02-17','2026-02-18',
                                       '2026-03-02','2026-05-05','2026-05-25','2026-05-01','2026-06-03')
                THEN 1 ELSE 0 END AS 휴일,
           CASE WHEN 강수량_mm = 0 THEN '1) 0'
                WHEN 강수량_mm <= 5 THEN '2) 0~5mm'
                WHEN 강수량_mm <= 20 THEN '3) 5~20mm'
                ELSE '4) 20mm+' END AS 강수구간
    FROM daily
),
rel AS (
    SELECT *,
           이용건수 * 1.0 / AVG(이용건수) OVER (PARTITION BY 월, 휴일) AS 상대이용
    FROM base
)
SELECT 강수구간, COUNT(*) AS 일수, ROUND(AVG(상대이용), 3) AS 상대이용
FROM rel
GROUP BY 강수구간
ORDER BY 강수구간
""")

,강수구간,일수,상대이용
0,1) 0,143,1.065
1,2) 0~5mm,27,0.847
2,3) 5~20mm,9,0.611
3,4) 20mm+,2,0.202


In [4]:
q("""
SELECT date(대여일자) AS 날짜, 요일, 강수량_mm, 이용건수,
       LAG(이용건수) OVER (ORDER BY 대여일자) AS 전날,
       ROUND(이용건수 * 1.0 / LAG(이용건수) OVER (ORDER BY 대여일자) - 1, 3) AS 변화율
FROM daily
ORDER BY 변화율 NULLS LAST
LIMIT 10
""")

,날짜,요일,강수량_mm,이용건수,전날,변화율
0,2026-05-20,Wednesday,63.1,12362,152640,-0.919
1,2026-03-02,Monday,12.2,9975,68890,-0.855
2,2026-04-09,Thursday,18.9,37251,131740,-0.717
3,2026-05-03,Sunday,14.8,31608,111557,-0.717
4,2026-06-20,Saturday,71.7,34374,90077,-0.618
5,2026-03-18,Wednesday,4.8,51911,107482,-0.517
6,2026-01-24,Saturday,0.0,25288,51507,-0.509
7,2026-01-10,Saturday,1.7,35620,70746,-0.497
8,2026-06-19,Friday,8.0,90077,148819,-0.395
9,2026-02-07,Saturday,0.0,33305,54882,-0.393
